# 🗜️ HarMeme OCR — Notebook 3: Merger
Merges all 4 chunk `.pkl` files into one consolidated `ocr_harmeme_full.pkl`
and compresses it to a `.zip` for upload as a Kaggle dataset.

**Prerequisites:** Attach the dataset containing all 4 chunk pkls as `harmeme-ocr-chunks`.


In [ ]:
import pickle, zipfile
import pandas as pd
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
# Dataset containing ocr_chunk_harmeme_0.pkl … ocr_chunk_harmeme_3.pkl
CHUNKS_DIR = Path("/kaggle/input/harmeme-ocr-chunks")
OUTPUT_DIR = Path("/kaggle/working")
N_CHUNKS   = 4

assert CHUNKS_DIR.exists(), f"❌ Chunks dataset not found at {CHUNKS_DIR}"

# Check which chunks are present
for i in range(N_CHUNKS):
    p = CHUNKS_DIR / f"ocr_chunk_harmeme_{i}.pkl"
    status = f"✅ {p.stat().st_size/1e6:.1f} MB" if p.exists() else "❌ MISSING"
    print(f"  Chunk {i}: {status}")


In [ ]:
# ── Merge all chunks ──────────────────────────────────────────────────────────
merged = {}

for chunk_id in range(N_CHUNKS):
    chunk_path = CHUNKS_DIR / f"ocr_chunk_harmeme_{chunk_id}.pkl"

    if not chunk_path.exists():
        print(f"⚠️  Chunk {chunk_id} missing — skipping.")
        continue

    with open(chunk_path, "rb") as f:
        chunk_data = pickle.load(f)

    merged.update(chunk_data)
    print(f"  Chunk {chunk_id}: {len(chunk_data):5d} entries merged | running total: {len(merged)}")

print(f"\n✅ Total merged entries: {len(merged)}")

# ── Save merged pkl ────────────────────────────────────────────────────────────
MERGED_PKL = OUTPUT_DIR / "ocr_harmeme_full.pkl"
with open(MERGED_PKL, "wb") as f:
    pickle.dump(merged, f)
print(f"✅ PKL saved → {MERGED_PKL}  ({MERGED_PKL.stat().st_size/1e6:.1f} MB)")

# ── Compress to zip ────────────────────────────────────────────────────────────
MERGED_ZIP = OUTPUT_DIR / "ocr_harmeme_full.zip"
with zipfile.ZipFile(MERGED_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(MERGED_PKL, "ocr_harmeme_full.pkl")
print(f"✅ ZIP saved → {MERGED_ZIP}  ({MERGED_ZIP.stat().st_size/1e6:.1f} MB)")

# ── Sanity check ──────────────────────────────────────────────────────────────
print("\nSample entries:")
for k, v in list(merged.items())[:4]:
    print(f"  {k:30s}  words={v['words'][:4]}  boxes={v['boxes'][:2]}")


In [ ]:
# ── Verify coverage against original annotations ──────────────────────────────
import json

HARMC_ROOT = Path("/kaggle/input/harmeme/MINI_PROJECT_2/Harm-C/datasets/memes/defaults")
HARMP_ROOT = Path("/kaggle/input/harmeme/MINI_PROJECT_2/Harm-P/datasets/memes/defaults")

all_fnames = set()
for root in [HARMC_ROOT, HARMP_ROOT]:
    for jsonl_file in sorted((root / "annotations").glob("*.jsonl")):
        with open(jsonl_file) as f:
            for line in f:
                if line.strip():
                    r = json.loads(line)
                    fname = str(r.get("image","")).replace("img/","").split("/")[-1]
                    all_fnames.add(fname)

covered  = all_fnames & set(merged.keys())
missing  = all_fnames - set(merged.keys())

print(f"Total unique images in annotations : {len(all_fnames)}")
print(f"Covered by OCR                     : {len(covered)}")
print(f"Missing from OCR                   : {len(missing)}")
if missing:
    print(f"  Missing sample: {list(missing)[:5]}")
else:
    print("✅ Full coverage — all images have OCR entries.")


## 📋 How to use in your ISSUES + LiLT main notebook

After uploading `ocr_harmeme_full.zip` as a Kaggle dataset (e.g. `harmeme-ocr`),
add the following lines **right after** your HMC `ocr_data` is loaded:

```python
import pickle, zipfile
from pathlib import Path

# ── Load and merge HarMeme OCR ────────────────────────────────────
harmeme_zip = "/kaggle/input/harmeme-ocr/ocr_harmeme_full.zip"
with zipfile.ZipFile(harmeme_zip, "r") as zf:
    with zf.open("ocr_harmeme_full.pkl") as pf:
        harmeme_ocr = pickle.load(pf)

# Normalise keys to bare filename just in case
harmeme_ocr = {str(k).split("/")[-1]: v for k, v in harmeme_ocr.items()}

ocr_data.update(harmeme_ocr)
print(f"✅ HarMeme OCR merged — total entries: {len(ocr_data)}")
```

Then add HarMeme splits to your `SPLITS` dict:

```python
HARMC_ANNOT = Path("/kaggle/input/harmeme/MINI_PROJECT_2/Harm-C/datasets/memes/defaults/annotations")
HARMP_ANNOT = Path("/kaggle/input/harmeme/MINI_PROJECT_2/Harm-P/datasets/memes/defaults/annotations")

SPLITS["harmc_train"] = HARMC_ANNOT / "train.jsonl"
SPLITS["harmc_val"]   = HARMC_ANNOT / "val.jsonl"
SPLITS["harmp_train"] = HARMP_ANNOT / "train.jsonl"
SPLITS["harmp_val"]   = HARMP_ANNOT / "val.jsonl"
```

**Label mapping** — HarMeme uses string labels, add this inside your split-loading loop:
```python
if "labels" in df.columns and "label" not in df.columns:
    df["label"] = df["labels"].apply(
        lambda x: 0 if (x[0] if isinstance(x, list) else x) == "not harmful" else 1
    )
if "image" in df.columns and "img" not in df.columns:
    df["img"] = df["image"]
```
